In [1]:
%load_ext autoreload
%autoreload 2
from core.nn.AudioMNIST import AudioMnistDataset
from datasets import load_metric, Dataset, DatasetDict, Audio
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer, AutoFeatureExtractor
import torch
model_checkpoint = "facebook/wav2vec2-base"
from safetensors import safe_open

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
wav_dir = "/media/data/AudioMNIST"
csv_dir = "/media/data/resnew/8000_20000_50"
MAX_DURATION = 1.5

In [6]:
model_checkpoint = "facebook/wav2vec2-base"
wavef_model_digit = AutoModelForAudioClassification.from_pretrained(model_checkpoint, num_labels=10).to(device)
mouse_model_digit = AutoModelForAudioClassification.from_pretrained(model_checkpoint, num_labels=10).to(device)
wavef_model_speaker = AutoModelForAudioClassification.from_pretrained(model_checkpoint, num_labels=61).to(device)
mouse_model_speaker = AutoModelForAudioClassification.from_pretrained(model_checkpoint, num_labels=61).to(device)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_checkpoint)
dataset_pt = AudioMnistDataset(wav_dir, csv_dir)
datasetDict = dataset_pt.split_dataset()
metric = load_metric("accuracy")

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2

AttributeError: 'AudioMnistDataset' object has no attribute 'to'

In [8]:
def new_forward(self, input_values):
        hidden_states = input_values

        # make sure hidden_states require grad for gradient_checkpointing
        if self._requires_grad and self.training:
            hidden_states.requires_grad = True

        for conv_layer in self.conv_layers:
            if self._requires_grad and self.gradient_checkpointing and self.training:
                hidden_states = self._gradient_checkpointing_func(
                    conv_layer.__call__,
                    hidden_states,
                )
            else:
                hidden_states = conv_layer(hidden_states)

        return hidden_states
mouse_model_digit.wav2vec2.feature_extractor.forward = new_forward.__get__(mouse_model_digit.wav2vec2.feature_extractor, mouse_model_digit.wav2vec2.feature_extractor.__class__)
mouse_model_digit.wav2vec2.feature_extractor.conv_layers[0].conv = torch.nn.Conv1d(2, 512, kernel_size=(10,), stride=(5,), bias=False).to(device)

mouse_model_speaker.wav2vec2.feature_extractor.forward = new_forward.__get__(mouse_model_speaker.wav2vec2.feature_extractor, mouse_model_speaker.wav2vec2.feature_extractor.__class__)
mouse_model_speaker.wav2vec2.feature_extractor.conv_layers[0].conv = torch.nn.Conv1d(2, 512, kernel_size=(10,), stride=(5,), bias=False).to(device)

In [10]:
datasetDict

(<torch.utils.data.dataset.Subset at 0x7c1bdf53dfa0>,
 <torch.utils.data.dataset.Subset at 0x7c1aea6d11c0>)

In [9]:
# keep only the waveform_audio and digit_target columns
def preprocess_function(examples):
    return feature_extractor(
        examples, 
        padding='max_length',
        sampling_rate=feature_extractor.sampling_rate, 
        max_length=int(feature_extractor.sampling_rate * MAX_DURATION), 
        truncation=True, 
    )

def preprocess_function_mouse(examples):
    return feature_extractor(
        examples, 
        padding='max_length',
        sampling_rate=feature_extractor.sampling_rate, 
        max_length=int(feature_extractor.sampling_rate * MAX_DURATION), 
        truncation=True, 
    )

WV_DG_dataset = datasetDict.map(preprocess_function, input_columns=["original_audio"], batched=True).remove_columns(["mouse_audio", "speaker_target", "original_audio"]).rename_column("digit_target", "labels")

AttributeError: 'tuple' object has no attribute 'map'

In [26]:
WV_SP_dataset = datasetDict.map(preprocess_function, input_columns=["original_audio"], batched=True).remove_columns(["mouse_audio", "digit_target", "original_audio"]).rename_column("speaker_target", "labels")

In [27]:
MS_DG_dataset = datasetDict.map(preprocess_function_mouse, input_columns=["mouse_audio"], batched=False).remove_columns(["original_audio", "speaker_target", "mouse_audio"]).rename_column("digit_target", "labels")

Map:   0%|          | 0/25500 [00:00<?, ? examples/s]

Map:   0%|          | 0/3750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [28]:
MS_SP_dataset = datasetDict.map(preprocess_function_mouse, input_columns=["mouse_audio"], batched=False).remove_columns(["original_audio", "digit_target", "mouse_audio"]).rename_column("speaker_target", "labels")

In [29]:
args = TrainingArguments(
    "test",
    evaluation_strategy = "epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=32,
    num_train_epochs=20,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    remove_unused_columns=False
)

/opt/mambaforge/envs/micemouse/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [30]:
import numpy as np

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [31]:
len(MS_DG_dataset["train"][0]['input_values'][0])

24000

In [32]:
mouse_model_digit(torch.zeros(32, 2, 16000).to(device))

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
         -0.0214,  0.0835],
        [ 0.0192, -0.0143,  0.0342, -0.0675,  0.0544, -0.0272, -0.0255,  0.0041,
    

In [33]:
trainer = Trainer(
    mouse_model_digit,
    args,
    train_dataset=MS_DG_dataset["train"],
    eval_dataset=MS_DG_dataset["test"],
    compute_metrics=compute_metrics
)

In [36]:
run_name = 23232
mouse_model_digit.save_pretrained(f"/media/data/models/{run_name}/mouse_model_digit")

In [34]:
trainer.train()

  0%|          | 0/15940 [00:00<?, ?it/s]

/opt/mambaforge/envs/micemouse/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 2.3046, 'grad_norm': 1.2576056718826294, 'learning_rate': 1.2547051442910918e-07, 'epoch': 0.01}
{'loss': 2.3032, 'grad_norm': 1.5449180603027344, 'learning_rate': 2.5094102885821835e-07, 'epoch': 0.03}
{'loss': 2.3033, 'grad_norm': 1.9310301542282104, 'learning_rate': 3.764115432873275e-07, 'epoch': 0.04}
{'loss': 2.3061, 'grad_norm': 1.4290696382522583, 'learning_rate': 5.018820577164367e-07, 'epoch': 0.05}
{'loss': 2.3059, 'grad_norm': 1.2912999391555786, 'learning_rate': 6.273525721455459e-07, 'epoch': 0.06}
{'loss': 2.3031, 'grad_norm': 1.119020938873291, 'learning_rate': 7.52823086574655e-07, 'epoch': 0.08}
{'loss': 2.302, 'grad_norm': 1.040928602218628, 'learning_rate': 8.782936010037642e-07, 'epoch': 0.09}
{'loss': 2.3045, 'grad_norm': 1.0774043798446655, 'learning_rate': 1.0037641154328734e-06, 'epoch': 0.1}
{'loss': 2.3002, 'grad_norm': 1.0763766765594482, 'learning_rate': 1.1292346298619825e-06, 'epoch': 0.11}
{'loss': 2.3026, 'grad_norm': 1.4583451747894287, 'learn

  0%|          | 0/118 [00:00<?, ?it/s]

{'eval_loss': 2.138185977935791, 'eval_accuracy': 0.21093333333333333, 'eval_runtime': 39.3244, 'eval_samples_per_second': 95.361, 'eval_steps_per_second': 3.001, 'epoch': 1.0}


/opt/mambaforge/envs/micemouse/lib/python3.12/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 2.196, 'grad_norm': 8.378142356872559, 'learning_rate': 1.0037641154328735e-05, 'epoch': 1.0}
{'loss': 2.1913, 'grad_norm': 4.3035993576049805, 'learning_rate': 1.0163111668757844e-05, 'epoch': 1.02}
{'loss': 2.1699, 'grad_norm': 6.159455299377441, 'learning_rate': 1.0288582183186952e-05, 'epoch': 1.03}
{'loss': 2.1647, 'grad_norm': 4.776688575744629, 'learning_rate': 1.0414052697616062e-05, 'epoch': 1.04}
{'loss': 2.1338, 'grad_norm': 6.847347736358643, 'learning_rate': 1.053952321204517e-05, 'epoch': 1.05}
{'loss': 2.141, 'grad_norm': 5.297712326049805, 'learning_rate': 1.066499372647428e-05, 'epoch': 1.07}
{'loss': 2.1345, 'grad_norm': 4.750694274902344, 'learning_rate': 1.079046424090339e-05, 'epoch': 1.08}
{'loss': 2.1826, 'grad_norm': 10.893129348754883, 'learning_rate': 1.0915934755332498e-05, 'epoch': 1.09}
{'loss': 2.1732, 'grad_norm': 10.912020683288574, 'learning_rate': 1.1041405269761607e-05, 'epoch': 1.1}
{'loss': 2.1439, 'grad_norm': 4.852091312408447, 'learning_

KeyboardInterrupt: 

In [35]:
trainer.evaluate()

  0%|          | 0/118 [00:00<?, ?it/s]

{'eval_loss': 2.1782407760620117, 'eval_accuracy': 0.21706666666666666, 'eval_runtime': 39.7498, 'eval_samples_per_second': 94.34, 'eval_steps_per_second': 2.969, 'epoch': 1.63}


{'eval_loss': 2.1782407760620117,
 'eval_accuracy': 0.21706666666666666,
 'eval_runtime': 39.7498,
 'eval_samples_per_second': 94.34,
 'eval_steps_per_second': 2.969,
 'epoch': 1.6260978670012547}